# Hawkes-on-LOB: Multivariate Hawkes Process Fit

This notebook fits a 4-dimensional Hawkes process with exponential kernels
to LOBSTER limit order book event data (GOOG, 2012-06-21), then validates
the fit using the time-rescaling theorem.

**Event types:**
- **MB** — Market Buy (aggressive buy executions)
- **MS** — Market Sell (aggressive sell executions)
- **LA_B** — Limit Add Buy (new buy limit orders)
- **LA_S** — Limit Add Sell (new sell limit orders)

**Model:**

$$\lambda_i(t) = \mu_i + \sum_j \alpha_{ij} \sum_{t_{jk}<t} \beta \cdot e^{-\beta(t-t_{jk})}$$

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from src.data import load_lobster, EVENT_LABELS
from src.model import fit_hawkes
from src.validation import time_rescaling, gof_test

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load LOBSTER Data

In [ ]:
MESSAGE_CSV = "../data/GOOG_2012-06-21_34200000_57600000_message_10.csv"

timestamps = load_lobster(MESSAGE_CSV)

### Event arrival overview

Quick sanity check: plot cumulative event counts and a 1-minute
binned arrival rate for each dimension.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

colors = {"MB": "#e63946", "MS": "#457b9d", "LA_B": "#2a9d8f", "LA_S": "#e9c46a"}

# Cumulative counts
ax = axes[0]
for label in EVENT_LABELS:
    ts = timestamps[label]
    ax.plot(ts / 3600, np.arange(1, len(ts) + 1), label=label,
            color=colors[label], linewidth=0.8)
ax.set_ylabel("Cumulative events")
ax.legend(loc="upper left", fontsize=9)
ax.set_title("GOOG — 2012-06-21 — Event Arrivals")

# 1-minute binned rates
ax = axes[1]
T_max = max(ts.max() for ts in timestamps.values())
bins = np.arange(0, T_max + 60, 60)
for label in EVENT_LABELS:
    counts, _ = np.histogram(timestamps[label], bins=bins)
    ax.plot(bins[:-1] / 3600, counts, label=label,
            color=colors[label], alpha=0.7, linewidth=0.6)
ax.set_ylabel("Events / minute")
ax.set_xlabel("Hours since market open")
ax.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.savefig("../outputs/event_arrivals.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"Trading day span: {T_max/3600:.2f} hours")

## 2. Fit Multivariate Hawkes Process

Using `tick.hawkes.HawkesExpKern` with a shared exponential decay β = 1.0.
This is a reasonable starting point — at β = 1 the kernel has a half-life
of ~0.7 seconds.

In [ ]:
result = fit_hawkes(timestamps, decay=1.0)

### Fitted Parameters

In [ ]:
import pandas as pd

labels = result["labels"]

print("=== Baseline Intensities μ (events/sec) ===")
mu_df = pd.DataFrame({"μ": result["mu"]}, index=labels)
display(mu_df)

print("\n=== Excitation Matrix α (row=target, col=source) ===")
alpha_df = pd.DataFrame(result["alpha"], index=labels, columns=labels)
display(alpha_df)

print(f"\nSpectral radius ρ(α) = {result['spectral_radius']:.4f}")
if result["spectral_radius"] < 1:
    print("✓ Process is stable (ρ < 1)")
else:
    print("✗ Process is UNSTABLE (ρ ≥ 1) — model is misspecified")

In [ ]:
np.savez("../outputs/fit_params.npz",
         mu=result["mu"],
         alpha=result["alpha"],
         decay=np.array([result["decay"]]),
         spectral_radius=np.array([result["spectral_radius"]]),
         labels=np.array(labels))
print("Saved fit parameters to outputs/fit_params.npz")

### Excitation Matrix Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

alpha = result["alpha"]

# Log scale for better contrast (add small epsilon to avoid log(0))
import matplotlib.colors as mcolors
vmin = max(alpha[alpha > 0].min() * 0.5, 1e-6) if (alpha > 0).any() else 1e-6
vmax = alpha.max() * 1.2

im = ax.imshow(alpha, cmap="YlOrRd",
               norm=mcolors.LogNorm(vmin=vmin, vmax=vmax))

# Annotate cells
for i in range(4):
    for j in range(4):
        val = alpha[i, j]
        color = "white" if val > vmax * 0.3 else "black"
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                fontsize=10, color=color, fontweight="bold")

ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.set_xticklabels(labels, fontsize=11)
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel("Source (j — who caused it)", fontsize=12)
ax.set_ylabel("Target (i — who gets excited)", fontsize=12)
ax.set_title("Excitation Matrix αᵢⱼ", fontsize=14)
plt.colorbar(im, ax=ax, label="αᵢⱼ (log scale)")

plt.tight_layout()
plt.savefig("../outputs/excitation_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Goodness-of-Fit Validation

### Time-Rescaling Theorem

If the model is correct, the compensator-transformed inter-event times
$\tau_{ik} = \Lambda_i(t_{ik}) - \Lambda_i(t_{i,k-1})$ should be
i.i.d. Exp(1). We test this with:

1. **KS test** against Exp(1) — tests the marginal distribution
2. **Ljung-Box** — tests serial independence (no residual clustering)

⚠️ **Note**: With the full dataset (~400k events), the compensator
computation is O(n²) and slow. We subsample to the first hour for
validation speed.

In [ ]:
# Subsample to first hour for tractable validation
T_VALIDATION = 3600.0  # first hour

ts_sub = {}
for label in EVENT_LABELS:
    mask = timestamps[label] < T_VALIDATION
    ts_sub[label] = timestamps[label][mask]
    print(f"  {label}: {mask.sum():,} events in first hour")

print("\nComputing compensator increments...")
transformed = time_rescaling(result, ts_sub)

print("\n=== Goodness-of-Fit Results ===")
gof = gof_test(transformed)

### QQ Plots: Transformed Times vs Exp(1)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for idx, label in enumerate(EVENT_LABELS):
    ax = axes[idx // 2, idx % 2]
    tau = transformed[label]
    if len(tau) == 0:
        ax.set_title(f"{label} — no data")
        continue

    # Sort transformed times for QQ
    tau_sorted = np.sort(tau)
    n = len(tau_sorted)
    # Theoretical Exp(1) quantiles
    theoretical = stats.expon.ppf((np.arange(1, n + 1) - 0.5) / n)

    ax.scatter(theoretical, tau_sorted, s=1, alpha=0.5,
              color=colors[label])

    # 45-degree reference line
    q_max = max(theoretical.max(), tau_sorted.max())
    ax.plot([0, q_max], [0, q_max], "k--", linewidth=0.8, alpha=0.6)

    ks_p = gof[label]["ks_p"]
    lb_p = gof[label]["lb_p"]
    ax.set_title(f"{label}  (KS p={ks_p:.3f}, LB p={lb_p:.3f})",
                fontsize=11)
    ax.set_xlabel("Exp(1) theoretical quantiles")
    ax.set_ylabel("Empirical quantiles")

plt.suptitle("QQ Plots — Time-Rescaled Inter-Event Times vs Exp(1)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("../outputs/qq_plots.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Summary

| Dimension | KS stat | KS p-value | LB stat | LB p-value | KS verdict | LB verdict |
|-----------|---------|------------|---------|------------|------------|------------|

*(Filled programmatically below)*

In [ ]:
print("\n" + "="*70)
print("VALIDATION SUMMARY")
print("="*70)
print(f"{"Dim":>5s} | {"KS D":>8s} | {"KS p":>8s} | {"LB Q":>8s} | {"LB p":>8s} | KS    | LB")
print("-"*70)
for label in EVENT_LABELS:
    g = gof[label]
    ks_v = "PASS" if g["ks_p"] > 0.05 else "FAIL"
    lb_v = "PASS" if g["lb_p"] > 0.05 else "FAIL"
    print(f'{label:>5s} | {g["ks_stat"]:8.4f} | {g["ks_p"]:8.4f} | '
          f'{g["lb_stat"]:8.2f} | {g["lb_p"]:8.4f} | {ks_v:5s} | {lb_v}')
print("="*70)
print("\nα = 0.05 significance level")
print(f"Spectral radius: {result['spectral_radius']:.4f}")